<a href="https://colab.research.google.com/github/izzat-ai/learning-ai/blob/main/scikit-learn/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Ushbu sahifada Pipeline - bir nechta preprocessing jarayonlarini birlashtirishni o'rganamiz**

In [1]:
import numpy as np
import pandas as pd
import sklearn

In [2]:
df = pd.DataFrame({
    "age": [20, 25, np.nan, 35, 40, 45, 50, 55],
    "salary": [2000, 3000, 4000, np.nan, 6000, 7000, 8000, 9000],
    "target": [0, 0, 0, 1, 1, 1, 1, 1]
})
df

,age,salary,target
0,20.0,2000.0,0
1,25.0,3000.0,0
2,NaN,4000.0,0
3,35.0,NaN,1
4,40.0,6000.0,1
5,45.0,7000.0,1
6,50.0,8000.0,1
7,55.0,9000.0,1


In [3]:
# X va y larni ajratish
X = df.drop('target', axis=1)
y = df['target']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("train shape:", X_train.shape)
print("test shape:", X_test.shape)

train shape: (6, 2)
test shape: (2, 2)


In [4]:
# kerakli paketlarni chaqirish va pipeline yaratish
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
])

In [5]:
pipe.fit(X_train, y_train)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()), ('model', LogisticRegression())])

In [6]:
# bashorat qilish
y_pred = pipe.predict(X_test)
y_pred

array([0, 1])

In [7]:
# boshqa DF bilan bashorat qilib ko'rish
new_student = pd.DataFrame({
    "age": [30],
    "salary": [5000]
})

student_predict = pipe.predict(new_student)
student_predict

array([1])

####**ColumnTransformer + Pipeline**

In [8]:
df2 = pd.DataFrame({
    "age": [20, 25, np.nan, 35, 40, 45, 50, 55, 60, 65],
    "salary": [2000, 3000, 4000, np.nan, 6000, 7000, 8000, 9000, 10000, 11000],
    "city": [
        "Tashkent", "Samarkand", "Bukhara", "Tashkent",
        "Samarkand", "Bukhara", "Tashkent", "Samarkand",
        "Bukhara", "Tashkent"
    ],
    "gender": [
        "Male", "Female", "Male", "Female",
        "Male", "Female", "Male", "Female",
        "Male", "Female"
    ],
    "target": [0, 0, 0, 1, 1, 1, 1, 1, 1, 1]
})
df2

,age,salary,city,gender,target
0,20.0,2000.0,Tashkent,Male,0
1,25.0,3000.0,Samarkand,Female,0
2,NaN,4000.0,Bukhara,Male,0
3,35.0,NaN,Tashkent,Female,1
4,40.0,6000.0,Samarkand,Male,1
5,45.0,7000.0,Bukhara,Female,1
6,50.0,8000.0,Tashkent,Male,1
7,55.0,9000.0,Samarkand,Female,1
8,60.0,10000.0,Bukhara,Male,1
9,65.0,11000.0,Tashkent,Female,1


In [10]:
# NaN larni aniqlash
df2.isnull().sum()

,0
age,1
salary,1
city,0
gender,0
target,0


In [11]:
# X va y larni ajratish
X = df2.drop('target', axis=1)
y = df2['target']

# train va testlarga bo'lish
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("train shape:", X_train.shape)
print("test shape:", X_test.shape)

train shape: (8, 4)
test shape: (2, 4)


In [12]:
# sonli va matnli ustunlarni olish
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()

print("sonli ustunlar:", num_cols)
print("matnli ustunlar:", cat_cols)

sonli ustunlar: ['age', 'salary']
matnli ustunlar: ['city', 'gender']


In [16]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [28]:
# sonli ustunlar uchun pipeline yaratish
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# kategoriyali ustunlar uchun pipeline yaratish
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

In [29]:
# ColumnTransformer yaratish va pipelinelarni birlashtirish
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

In [30]:
from sklearn.linear_model import LogisticRegression

# ColumnTransformer va modelni birlashtirish
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression())
])

In [31]:
# ma'lumotlarni tayyorlash va modelni o'qitish
full_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'salary']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['city', 'gender'])])),
                ('model', LogisticRegression())])

In [32]:
# bashorat qilish
y_pred = full_pipeline.predict(X_test)

In [33]:
y_pred

array([1, 1])

In [34]:
# baholash
full_pipeline.score(X_test, y_test)

0.5

In [35]:
new_data = pd.DataFrame({'age':[30], 'salary':[5000], 'city':["Khiva"], 'gender':["Male"]})
new_data

,age,salary,city,gender
0,30,5000,Khiva,Male


In [36]:
# yangi ma'lumotni bashorat qilish
full_pipeline.predict(new_data)

array([1])